# 12 — MLB Totals + Run-Line Model Lab

This notebook starts the next two betting functions without disturbing the existing moneyline pipeline.

It trains:

1. **Total runs models**: predicts `home_score + away_score`.
   - Standard regression candidates.
   - Poisson-style count-model candidates.
   - The notebook compares them and picks the best totals champion by validation error.
2. **Home margin regression**: predicts `home_score - away_score` for run-line/spread logic.
3. Optional **home -1.5 cover classifier** for quick diagnostics.

The production scorer can use the total-runs model against totals markets and the margin model against run-line/spread markets.


In [11]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except Exception as exc:
    HAS_XGB = False
    print("XGBoost not available:", exc)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [12]:
# Resolve project root from either notebooks/ or repo root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mlb_game_features.parquet"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
assert DATA_PATH.exists(), f"Missing feature parquet: {DATA_PATH}"

PROJECT_ROOT: c:\Users\rfo7799\Desktop\Git\TetheredAI\MLB\mlb_betting_app
DATA_PATH: c:\Users\rfo7799\Desktop\Git\TetheredAI\MLB\mlb_betting_app\data\processed\mlb_game_features.parquet


In [13]:
features = pd.read_parquet(DATA_PATH)
features["official_date"] = pd.to_datetime(features["official_date"], errors="coerce")
features["game_datetime_utc"] = pd.to_datetime(features["game_datetime_utc"], utc=True, errors="coerce")

print("Rows:", len(features))
print("Columns:", len(features.columns))
print("All date range:", features["official_date"].min(), "to", features["official_date"].max())

Rows: 9293
Columns: 4910
All date range: 2022-08-02 00:00:00 to 2026-09-22 00:00:00


## 1. Create totals and run-line targets

In [14]:
TARGET_TOTAL_RUNS = "target_total_runs"
TARGET_HOME_MARGIN = "target_home_margin"
TARGET_HOME_COVER_MINUS_1_5 = "target_home_cover_minus_1_5"

completed = features[
    features["home_score"].notna()
    & features["away_score"].notna()
].copy()

completed[TARGET_TOTAL_RUNS] = completed["home_score"].astype(float) + completed["away_score"].astype(float)
completed[TARGET_HOME_MARGIN] = completed["home_score"].astype(float) - completed["away_score"].astype(float)
completed[TARGET_HOME_COVER_MINUS_1_5] = (completed[TARGET_HOME_MARGIN] > 1.5).astype(int)

MIN_TRAIN_DATE = "2023-01-01"
completed = completed[completed["official_date"] >= MIN_TRAIN_DATE].copy()

print("Completed rows:", len(completed))
print("Completed range:", completed["official_date"].min(), "to", completed["official_date"].max())
print("Mean total runs:", completed[TARGET_TOTAL_RUNS].mean())
print("Mean home margin:", completed[TARGET_HOME_MARGIN].mean())
print("Home -1.5 cover rate:", completed[TARGET_HOME_COVER_MINUS_1_5].mean())

Completed rows: 8225
Completed range: 2023-03-30 00:00:00 to 2026-06-05 00:00:00
Mean total runs: 8.958662613981764
Mean home margin: 0.004741641337386018
Home -1.5 cover rate: 0.3574468085106383


## 2. Leakage-safe feature filtering

For totals and run-line models, postgame scores and result columns are leaks. We also avoid market/result/recommendation columns as model inputs.

In [15]:
LEAKY_COLS_EXACT = {
    "home_score", "away_score", "diff_score", "home_margin", "target_home_win",
    TARGET_TOTAL_RUNS, TARGET_HOME_MARGIN, TARGET_HOME_COVER_MINUS_1_5,
}

METADATA_COLS = {
    "game_pk", "run_id", "scored_at_utc", "official_date", "official_date_dt", "game_datetime_utc",
    "home_team_name", "away_team_name", "home_team_id", "away_team_id", "venue_name",
    "abstract_state", "detailed_state",
}

LEAKY_PATTERNS = [
    "winner", "winning", "losing", "final", "result", "outcome", "actual", "post_",
    "recommended", "edge", "kelly", "suggested", "market_", "moneyline", "price", "odds",
    "home_score", "away_score", "margin", "target_",
]

def is_probably_leaky_feature(col: str) -> bool:
    c = col.lower()
    if c in LEAKY_COLS_EXACT or c in METADATA_COLS:
        return True
    if c.endswith("_id"):
        return True
    return any(p in c for p in LEAKY_PATTERNS)

candidate_feature_cols = [
    c for c in completed.columns
    if c not in METADATA_COLS
    and c not in LEAKY_COLS_EXACT
    and pd.api.types.is_numeric_dtype(completed[c])
]

clean_feature_cols = [c for c in candidate_feature_cols if not is_probably_leaky_feature(c)]

print("Candidate feature count:", len(candidate_feature_cols))
print("Clean feature count:", len(clean_feature_cols))

removed = sorted(set(candidate_feature_cols) - set(clean_feature_cols))
display(pd.Series(removed, name="removed_features").head(100))

Candidate feature count: 4879
Clean feature count: 4862


0        away_moneyline_median
1     away_spread_price_median
2        diff_moneyline_median
3     diff_spread_price_median
4                 diff_team_id
5        home_moneyline_median
6     home_spread_price_median
7     market_away_implied_prob
8      market_away_no_vig_prob
9     market_home_implied_prob
10     market_home_no_vig_prob
11                  market_vig
12           over_price_median
13    probable_away_pitcher_id
14    probable_home_pitcher_id
15          under_price_median
16                    venue_id
Name: removed_features, dtype: object

## 3. Build focused feature families

In [16]:
def cols_containing_any(cols, terms):
    terms = [t.lower() for t in terms]
    return [c for c in cols if any(t in c.lower() for t in terms)]

def contains_any(col, terms):
    c = col.lower()
    return any(t in c for t in terms)

PITCHMIX_TERMS = ["pitchmix", "pitch_mix", "pitch_type", "pitchtype", "coarse_pitch", "arsenal", "matchup"]
BULLPEN_AVAIL_TERMS = ["bullpen_avail", "availability", "fatigue", "bullpen_pitches_last", "sc_bullpen_pitches_last", "diff_bullpen_pitches_last", "diff_sc_bullpen_pitches_last", "relievers_used", "back_to_back"]
BULLPEN_QUALITY_TERMS = ["bullpen_sc", "diff_bullpen_sc", "home_bullpen_sc", "away_bullpen_sc", "sc_bullpen"]
STATCAST_TERMS = ["statcast", "_sc_", "sc_", "batted_ball", "avg_ev", "distance", "hard_hit", "barrel"]

all_numeric_cols = [c for c in clean_feature_cols if c in completed.columns and pd.api.types.is_numeric_dtype(completed[c])]
statcast_cols = cols_containing_any(all_numeric_cols, STATCAST_TERMS)
pitchmix_cols = cols_containing_any(all_numeric_cols, PITCHMIX_TERMS)
bullpen_availability_cols = cols_containing_any(all_numeric_cols, BULLPEN_AVAIL_TERMS)
bullpen_quality_cols = cols_containing_any(all_numeric_cols, BULLPEN_QUALITY_TERMS)
old_statcast_cols = [c for c in statcast_cols if not contains_any(c, PITCHMIX_TERMS + BULLPEN_AVAIL_TERMS)]

feature_sets = {
    "old_statcast_no_pitchmix_bullpen": sorted(set(old_statcast_cols)),
    "true_pitchmix": sorted(set(pitchmix_cols)),
    "true_bullpen_availability": sorted(set(bullpen_availability_cols)),
    "pitchmix_plus_bullpen": sorted(set(pitchmix_cols + bullpen_quality_cols + bullpen_availability_cols)),
    "old_statcast_plus_pitchmix_bullpen": sorted(set(old_statcast_cols + pitchmix_cols + bullpen_availability_cols)),
    "all_clean": sorted(set(all_numeric_cols)),
}

feature_family_summary = pd.DataFrame([{"feature_family": k, "feature_count": len(v)} for k, v in feature_sets.items()]).sort_values("feature_count", ascending=False)
display(feature_family_summary)

MODEL_FEATURE_FAMILIES = [
    "old_statcast_no_pitchmix_bullpen",
    "pitchmix_plus_bullpen",
    "old_statcast_plus_pitchmix_bullpen",
    "all_clean",
]

,feature_family,feature_count
5,all_clean,4862
4,old_statcast_plus_pitchmix_bullpen,4191
3,pitchmix_plus_bullpen,3006
1,true_pitchmix,2583
0,old_statcast_no_pitchmix_bullpen,1503
2,true_bullpen_availability,105


## 4. Chronological train/test split

In [17]:
def chronological_split(df: pd.DataFrame, test_frac: float = 0.20):
    df = df.sort_values(["official_date", "game_datetime_utc", "game_pk"]).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_frac))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

train_df, test_df = chronological_split(completed, test_frac=0.20)
print("Train rows:", len(train_df), train_df["official_date"].min(), "to", train_df["official_date"].max())
print("Test rows:", len(test_df), test_df["official_date"].min(), "to", test_df["official_date"].max())

Train rows: 6580 2023-03-30 00:00:00 to 2025-08-06 00:00:00
Test rows: 1645 2025-08-06 00:00:00 to 2026-06-05 00:00:00


## 5. Model factories

In [18]:
def make_margin_regression_models():
    # Models for continuous margin regression.
    # Home margin can be negative, so Poisson is not appropriate here.
    models = {
        "random_forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=12, random_state=42, n_jobs=-1),
        "extra_trees": ExtraTreesRegressor(n_estimators=500, min_samples_leaf=10, random_state=42, n_jobs=-1),
        "hist_gbdt_squared_error": HistGradientBoostingRegressor(
            max_iter=350,
            learning_rate=0.035,
            l2_regularization=2.0,
            loss="squared_error",
            random_state=42,
        ),
    }
    if HAS_XGB:
        models["xgboost_squarederror"] = XGBRegressor(
            n_estimators=700,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.70,
            min_child_weight=20,
            reg_lambda=8.0,
            reg_alpha=0.25,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        )
    return models


def make_total_runs_models():
    # Total runs are nonnegative count-like outcomes, so include regular regression
    # candidates and Poisson-style candidates. The notebook compares all of them.
    models = make_margin_regression_models()

    # Poisson GLM. This can be useful for count data, but with thousands of features it can
    # be slower than tree models. The L2 alpha keeps it regularized.
    models["poisson_glm_l2"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", PoissonRegressor(alpha=1.0, max_iter=1000)),
    ])

    # Gradient boosting with Poisson loss. Good first Poisson-style nonlinear baseline.
    try:
        models["hist_gbdt_poisson"] = HistGradientBoostingRegressor(
            max_iter=350,
            learning_rate=0.035,
            l2_regularization=2.0,
            loss="poisson",
            random_state=42,
        )
    except TypeError as exc:
        print("HistGradientBoostingRegressor poisson loss not available in this sklearn version:", exc)

    if HAS_XGB:
        models["xgboost_poisson"] = XGBRegressor(
            n_estimators=700,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.70,
            min_child_weight=20,
            reg_lambda=8.0,
            reg_alpha=0.25,
            objective="count:poisson",
            random_state=42,
            n_jobs=-1,
        )

    return models


# Backward-compatible alias in case later cells refer to the old function name.
def make_regression_models():
    return make_margin_regression_models()


def make_classifier_models():
    models = {
        "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=12, random_state=42, n_jobs=-1),
        "extra_trees": ExtraTreesClassifier(n_estimators=500, min_samples_leaf=10, random_state=42, n_jobs=-1),
        "logit_l2": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, C=0.25, solver="lbfgs")),
        ]),
    }
    if HAS_XGB:
        models["xgboost"] = XGBClassifier(
            n_estimators=700,
            max_depth=2,
            learning_rate=0.025,
            subsample=0.85,
            colsample_bytree=0.70,
            min_child_weight=20,
            reg_lambda=8.0,
            reg_alpha=0.25,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )
    return models


def wrap_tree_model(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator),
    ])


def ensure_estimator(estimator):
    # Only wrap raw estimators. Leave Pipelines such as poisson_glm_l2 unchanged.
    return estimator if isinstance(estimator, Pipeline) else wrap_tree_model(estimator)


## 6. Train total-runs models, including Poisson candidates

Totals are nonnegative count-like outcomes. This section compares regular regression models against Poisson-style models and picks the best totals champion by validation MAE/RMSE. Poisson does **not** have to win; it is just another candidate.


In [19]:
def evaluate_regression(y_true, pred, include_poisson_metric: bool = False):
    y_true_arr = np.asarray(y_true, dtype=float)
    pred_arr = np.asarray(pred, dtype=float)

    mask = np.isfinite(y_true_arr) & np.isfinite(pred_arr)
    y_true_arr = y_true_arr[mask]
    pred_arr = pred_arr[mask]

    mse = mean_squared_error(y_true_arr, pred_arr)
    out = {
        "mae": float(mean_absolute_error(y_true_arr, pred_arr)),
        "rmse": float(np.sqrt(mse)),
        "r2": float(r2_score(y_true_arr, pred_arr)),
        "avg_pred": float(np.mean(pred_arr)),
        "actual_mean": float(np.mean(y_true_arr)),
    }

    if include_poisson_metric:
        # mean_poisson_deviance requires strictly positive predictions and nonnegative targets.
        pred_pos = np.clip(pred_arr, 1e-6, None)
        y_nonneg = np.clip(y_true_arr, 0.0, None)
        out["poisson_deviance"] = float(mean_poisson_deviance(y_nonneg, pred_pos))

    return out


totals_results = []
totals_fitted = {}
totals_preds = {}

y_train_total = train_df[TARGET_TOTAL_RUNS].astype(float)
y_test_total = test_df[TARGET_TOTAL_RUNS].astype(float)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue

    for model_name, base_model in make_total_runs_models().items():
        estimator = ensure_estimator(base_model)
        full_name = f"{family}__{model_name}"
        model_objective = "poisson" if "poisson" in model_name else "standard_regression"

        print("Training totals:", full_name, "features", len(cols), "objective", model_objective)
        estimator.fit(train_df[cols], y_train_total)

        # Total runs cannot be negative. Clip every model's prediction at zero for fair scoring.
        pred = estimator.predict(test_df[cols])
        pred = np.clip(np.asarray(pred, dtype=float), 0.0, None)

        metrics = evaluate_regression(y_test_total, pred, include_poisson_metric=True)
        totals_results.append({
            "model_name": full_name,
            "model_objective": model_objective,
            "feature_count": len(cols),
            "n_test": len(y_test_total),
            **metrics,
        })
        totals_fitted[full_name] = (estimator, cols)
        totals_preds[full_name] = pred

totals_results_df = (
    pd.DataFrame(totals_results)
    .sort_values(["mae", "rmse", "poisson_deviance"], na_position="last")
    .reset_index(drop=True)
)

display(totals_results_df.head(25))

print()
print("Best totals model selected by lowest MAE, then RMSE:")
display(totals_results_df.head(1))

print()
print("Poisson-style candidates only:")
display(totals_results_df[totals_results_df["model_objective"].eq("poisson")].head(15))


Training totals: old_statcast_no_pitchmix_bullpen__random_forest features 1503 objective standard_regression
Training totals: old_statcast_no_pitchmix_bullpen__extra_trees features 1503 objective standard_regression
Training totals: old_statcast_no_pitchmix_bullpen__hist_gbdt_squared_error features 1503 objective standard_regression
Training totals: old_statcast_no_pitchmix_bullpen__xgboost_squarederror features 1503 objective standard_regression
Training totals: old_statcast_no_pitchmix_bullpen__poisson_glm_l2 features 1503 objective poisson
Training totals: old_statcast_no_pitchmix_bullpen__hist_gbdt_poisson features 1503 objective poisson
Training totals: old_statcast_no_pitchmix_bullpen__xgboost_poisson features 1503 objective poisson
Training totals: pitchmix_plus_bullpen__random_forest features 3006 objective standard_regression
Training totals: pitchmix_plus_bullpen__extra_trees features 3006 objective standard_regression
Training totals: pitchmix_plus_bullpen__hist_gbdt_squared

C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training totals: all_clean__extra_trees features 4862 objective standard_regression


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training totals: all_clean__hist_gbdt_squared_error features 4862 objective standard_regression


...

,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
0,all_clean__random_forest,standard_regression,4862,1645,0.008820,0.227932,0.997389,8.946768,8.941033,0.011047
1,all_clean__hist_gbdt_squared_error,standard_regression,4862,1645,0.011397,0.229898,0.997344,8.947670,8.941033,0.011087
2,all_clean__hist_gbdt_poisson,poisson,4862,1645,0.012372,0.230711,0.997325,8.947953,8.941033,0.011100
3,all_clean__extra_trees,standard_regression,4862,1645,0.017581,0.233838,0.997252,8.953644,8.941033,0.011196
4,all_clean__xgboost_squarederror,standard_regression,4862,1645,0.041579,0.234700,0.997232,8.949527,8.941033,0.011468
5,all_clean__xgboost_poisson,poisson,4862,1645,0.073966,0.258525,0.996641,8.937472,8.941033,0.012155
6,all_clean__poisson_glm_l2,poisson,4862,1645,1.103621,1.442845,0.895373,8.847477,8.941033,0.286456
7,pitchmix_plus_bullpen__xgboost_squarederror,standard_regression,3006,1645,3.529508,4.452642,0.003589,8.787536,8.941033,2.236578
8,pitchmix_plus_bullpen__xgboost_poisson,poisson,3006,1645,3.530636,4.440138,0.009178,8.821990,8.941033,2.223862
9,old_statcast_plus_pitchmix_bullpen__extra_trees,standard_regression,4191,1645,3.533007,4.438698,0.009820,8.878954,8.941033,2.222295



Best totals model selected by lowest MAE, then RMSE:


,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
0,all_clean__random_forest,standard_regression,4862,1645,0.00882,0.227932,0.997389,8.946768,8.941033,0.011047



Poisson-style candidates only:


,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
2,all_clean__hist_gbdt_poisson,poisson,4862,1645,0.012372,0.230711,0.997325,8.947953,8.941033,0.011100
5,all_clean__xgboost_poisson,poisson,4862,1645,0.073966,0.258525,0.996641,8.937472,8.941033,0.012155
6,all_clean__poisson_glm_l2,poisson,4862,1645,1.103621,1.442845,0.895373,8.847477,8.941033,0.286456
8,pitchmix_plus_bullpen__xgboost_poisson,poisson,3006,1645,3.530636,4.440138,0.009178,8.821990,8.941033,2.223862
12,old_statcast_plus_pitchmix_bullpen__xgboost_po...,poisson,4191,1645,3.535275,4.441489,0.008575,8.834268,8.941033,2.225147
16,old_statcast_no_pitchmix_bullpen__xgboost_poisson,poisson,1503,1645,3.542826,4.443308,0.007762,8.869691,8.941033,2.227284
17,old_statcast_plus_pitchmix_bullpen__hist_gbdt_...,poisson,4191,1645,3.544239,4.511727,-0.023031,8.443808,8.941033,2.306577
20,pitchmix_plus_bullpen__hist_gbdt_poisson,poisson,3006,1645,3.556478,4.522305,-0.027833,8.444207,8.941033,2.314938
24,old_statcast_no_pitchmix_bullpen__hist_gbdt_po...,poisson,1503,1645,3.584074,4.531490,-0.032013,8.529920,8.941033,2.324576
25,old_statcast_no_pitchmix_bullpen__poisson_glm_l2,poisson,1503,1645,3.635878,4.599141,-0.063057,8.746229,8.941033,2.391703


### Poisson candidate notes

Poisson models are useful because total runs are nonnegative count outcomes. However, baseball totals are often overdispersed relative to a pure Poisson process, so the notebook treats Poisson as a candidate rather than forcing it. If a Poisson-style model wins by MAE/RMSE, export it; otherwise keep the best standard regression model.


## 7. Train home-margin regression models

In [20]:
margin_results = []
margin_fitted = {}
margin_preds = {}

y_train_margin = train_df[TARGET_HOME_MARGIN].astype(float)
y_test_margin = test_df[TARGET_HOME_MARGIN].astype(float)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, base_model in make_margin_regression_models().items():
        estimator = ensure_estimator(base_model)
        full_name = f"{family}__{model_name}"
        print("Training margin:", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_margin)
        pred = estimator.predict(test_df[cols])
        metrics = evaluate_regression(y_test_margin, pred)
        margin_results.append({"model_name": full_name, "feature_count": len(cols), "n_test": len(y_test_margin), **metrics})
        margin_fitted[full_name] = (estimator, cols)
        margin_preds[full_name] = pred

margin_results_df = pd.DataFrame(margin_results).sort_values(["mae", "rmse"]).reset_index(drop=True)
display(margin_results_df.head(20))


Training margin: old_statcast_no_pitchmix_bullpen__random_forest features 1503
Training margin: old_statcast_no_pitchmix_bullpen__extra_trees features 1503
Training margin: old_statcast_no_pitchmix_bullpen__hist_gbdt_squared_error features 1503
Training margin: old_statcast_no_pitchmix_bullpen__xgboost_squarederror features 1503
Training margin: pitchmix_plus_bullpen__random_forest features 3006
Training margin: pitchmix_plus_bullpen__extra_trees features 3006
Training margin: pitchmix_plus_bullpen__hist_gbdt_squared_error features 3006
Training margin: pitchmix_plus_bullpen__xgboost_squarederror features 3006
Training margin: old_statcast_plus_pitchmix_bullpen__random_forest features 4191
Training margin: old_statcast_plus_pitchmix_bullpen__extra_trees features 4191
Training margin: old_statcast_plus_pitchmix_bullpen__hist_gbdt_squared_error features 4191
Training margin: old_statcast_plus_pitchmix_bullpen__xgboost_squarederror features 4191
Training margin: all_clean__random_forest f

C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training margin: all_clean__extra_trees features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training margin: all_clean__hist_gbdt_squared_error features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training margin: all_clean__xgboost_squarederror features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,model_name,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean
0,all_clean__random_forest,4862,1645,3.499817,4.439886,0.035452,0.057553,0.054711
1,old_statcast_plus_pitchmix_bullpen__random_forest,4191,1645,3.504739,4.438301,0.036141,0.064521,0.054711
2,all_clean__xgboost_squarederror,4862,1645,3.505695,4.458715,0.027254,0.088757,0.054711
3,old_statcast_plus_pitchmix_bullpen__xgboost_sq...,4191,1645,3.507559,4.455825,0.028514,0.098443,0.054711
4,old_statcast_no_pitchmix_bullpen__xgboost_squa...,1503,1645,3.507894,4.458506,0.027345,0.057705,0.054711
5,pitchmix_plus_bullpen__xgboost_squarederror,3006,1645,3.508535,4.452380,0.030016,0.077336,0.054711
6,all_clean__extra_trees,4862,1645,3.510077,4.454172,0.029235,0.074646,0.054711
7,old_statcast_no_pitchmix_bullpen__random_forest,1503,1645,3.513636,4.450132,0.030995,0.045222,0.054711
8,pitchmix_plus_bullpen__random_forest,3006,1645,3.515408,4.451433,0.030429,0.037619,0.054711
9,old_statcast_no_pitchmix_bullpen__extra_trees,1503,1645,3.519787,4.455714,0.028563,0.066763,0.054711


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

y_train_margin = train_df[TARGET_HOME_MARGIN].astype(float)
y_test_margin = test_df[TARGET_HOME_MARGIN].astype(float)

baseline_margin_pred = np.repeat(y_train_margin.mean(), len(y_test_margin))

baseline_margin_metrics = {
    "model_name": "constant_train_home_margin",
    "mae": mean_absolute_error(y_test_margin, baseline_margin_pred),
    "rmse": np.sqrt(mean_squared_error(y_test_margin, baseline_margin_pred)),
    "r2": r2_score(y_test_margin, baseline_margin_pred),
    "avg_pred": baseline_margin_pred.mean(),
    "actual_mean": y_test_margin.mean(),
}

display(pd.DataFrame([baseline_margin_metrics]))

,model_name,mae,rmse,r2,avg_pred,actual_mean
0,constant_train_home_margin,3.600485,4.521176,-0.000191,-0.007751,0.054711


## 8. Optional run-line cover classifier diagnostic

In [22]:
cover_results = []
cover_fitted = {}
cover_preds = {}

y_train_cover = train_df[TARGET_HOME_COVER_MINUS_1_5].astype(int)
y_test_cover = test_df[TARGET_HOME_COVER_MINUS_1_5].astype(int)

for family in MODEL_FEATURE_FAMILIES:
    cols = [c for c in feature_sets[family] if c in train_df.columns]
    if not cols:
        continue
    for model_name, model in make_classifier_models().items():
        estimator = model if isinstance(model, Pipeline) else wrap_tree_model(model)
        full_name = f"{family}__{model_name}"
        print("Training cover classifier:", full_name, "features", len(cols))
        estimator.fit(train_df[cols], y_train_cover)
        p = estimator.predict_proba(test_df[cols])[:, 1]
        cover_results.append({
            "model_name": full_name,
            "feature_count": len(cols),
            "n_test": len(y_test_cover),
            "avg_pred": float(np.mean(p)),
            "actual_rate": float(y_test_cover.mean()),
            "log_loss": float(log_loss(y_test_cover, p)),
            "brier": float(brier_score_loss(y_test_cover, p)),
            "roc_auc": float(roc_auc_score(y_test_cover, p)),
            "accuracy_50pct": float(accuracy_score(y_test_cover, p >= 0.5)),
        })
        cover_fitted[full_name] = (estimator, cols)
        cover_preds[full_name] = p

cover_results_df = pd.DataFrame(cover_results).sort_values(["log_loss", "brier"]).reset_index(drop=True)
display(cover_results_df.head(20))

Training cover classifier: old_statcast_no_pitchmix_bullpen__random_forest features 1503
Training cover classifier: old_statcast_no_pitchmix_bullpen__extra_trees features 1503
Training cover classifier: old_statcast_no_pitchmix_bullpen__logit_l2 features 1503
Training cover classifier: old_statcast_no_pitchmix_bullpen__xgboost features 1503
Training cover classifier: pitchmix_plus_bullpen__random_forest features 3006
Training cover classifier: pitchmix_plus_bullpen__extra_trees features 3006
Training cover classifier: pitchmix_plus_bullpen__logit_l2 features 3006
Training cover classifier: pitchmix_plus_bullpen__xgboost features 3006
Training cover classifier: old_statcast_plus_pitchmix_bullpen__random_forest features 4191
Training cover classifier: old_statcast_plus_pitchmix_bullpen__extra_trees features 4191
Training cover classifier: old_statcast_plus_pitchmix_bullpen__logit_l2 features 4191
Training cover classifier: old_statcast_plus_pitchmix_bullpen__xgboost features 4191
Trainin

C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training cover classifier: all_clean__extra_trees features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training cover classifier: all_clean__logit_l2 features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Training cover classifier: all_clean__xgboost features 4862


C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\rfo7799\AppData\Roaming\Python\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['away_spread_median' 'book_count_h2h_away' 'book_count_h2h_home'
 'diff_spread_median' 'home_spread_median' 'total_points_median']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,model_name,feature_count,n_test,avg_pred,actual_rate,log_loss,brier,roc_auc,accuracy_50pct
0,all_clean__xgboost,4862,1645,0.359992,0.361702,0.627844,0.218751,0.631068,0.654711
1,old_statcast_no_pitchmix_bullpen__random_forest,1503,1645,0.360310,0.361702,0.644314,0.226258,0.579888,0.647416
2,old_statcast_plus_pitchmix_bullpen__xgboost,4191,1645,0.363592,0.361702,0.644456,0.226295,0.586335,0.641337
3,pitchmix_plus_bullpen__xgboost,3006,1645,0.361662,0.361702,0.645515,0.226649,0.580354,0.641337
4,old_statcast_no_pitchmix_bullpen__xgboost,1503,1645,0.360359,0.361702,0.645982,0.227111,0.584943,0.631003
5,old_statcast_plus_pitchmix_bullpen__random_forest,4191,1645,0.360904,0.361702,0.646092,0.226997,0.571198,0.643161
6,pitchmix_plus_bullpen__random_forest,3006,1645,0.362502,0.361702,0.646216,0.227032,0.572560,0.643769
7,all_clean__extra_trees,4862,1645,0.359834,0.361702,0.646296,0.227107,0.569613,0.641945
8,old_statcast_no_pitchmix_bullpen__extra_trees,1503,1645,0.360810,0.361702,0.646353,0.227164,0.570852,0.641337
9,old_statcast_plus_pitchmix_bullpen__extra_trees,4191,1645,0.360174,0.361702,0.646914,0.227440,0.568756,0.641945


## 9. Residual diagnostics for champion regressors

In [23]:
TOTALS_CHAMPION_NAME = totals_results_df.iloc[0]["model_name"]
MARGIN_CHAMPION_NAME = margin_results_df.iloc[0]["model_name"]

print("Totals champion:", TOTALS_CHAMPION_NAME)
display(totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)])

print("Margin champion:", MARGIN_CHAMPION_NAME)
display(margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)])

total_resid = y_test_total.values - totals_preds[TOTALS_CHAMPION_NAME]
margin_resid = y_test_margin.values - margin_preds[MARGIN_CHAMPION_NAME]

resid_summary = pd.DataFrame([
    {"target": "total_runs", "residual_mean": total_resid.mean(), "residual_std": total_resid.std(ddof=1), "p10": np.quantile(total_resid, 0.10), "p50": np.quantile(total_resid, 0.50), "p90": np.quantile(total_resid, 0.90)},
    {"target": "home_margin", "residual_mean": margin_resid.mean(), "residual_std": margin_resid.std(ddof=1), "p10": np.quantile(margin_resid, 0.10), "p50": np.quantile(margin_resid, 0.50), "p90": np.quantile(margin_resid, 0.90)},
])
display(resid_summary)

print()
print("Interpretation:")
print("- residual_mean near 0 means the model is not systematically high/low.")
print("- residual_std is used by the scoring script to approximate over/under and run-line probabilities.")


Totals champion: all_clean__random_forest


,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
0,all_clean__random_forest,standard_regression,4862,1645,0.00882,0.227932,0.997389,8.946768,8.941033,0.011047


Margin champion: all_clean__random_forest


,model_name,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean
0,all_clean__random_forest,4862,1645,3.499817,4.439886,0.035452,0.057553,0.054711


,target,residual_mean,residual_std,p10,p50,p90
0,total_runs,-0.005735,0.227929,0.000000,0.000000,0.000000
1,home_margin,-0.002841,4.441235,-5.646552,0.282951,5.371311



Interpretation:
- residual_mean near 0 means the model is not systematically high/low.
- residual_std is used by the scoring script to approximate over/under and run-line probabilities.


In [24]:
TOTALS_CHAMPION_NAME = 'old_statcast_plus_pitchmix_bullpen__random_forest'
MARGIN_CHAMPION_NAME = 'old_statcast_plus_pitchmix_bullpen__random_forest'
#TOTALS_CHAMPION_NAME = totals_results_df.iloc[0]["model_name"]
#MARGIN_CHAMPION_NAME = margin_results_df.iloc[0]["model_name"]

print("Totals champion:", TOTALS_CHAMPION_NAME)
display(totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)])

print("Margin champion:", MARGIN_CHAMPION_NAME)
display(margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)])

total_resid = y_test_total.values - totals_preds[TOTALS_CHAMPION_NAME]
margin_resid = y_test_margin.values - margin_preds[MARGIN_CHAMPION_NAME]

resid_summary = pd.DataFrame([
    {"target": "total_runs", "residual_mean": total_resid.mean(), "residual_std": total_resid.std(ddof=1), "p10": np.quantile(total_resid, 0.10), "p50": np.quantile(total_resid, 0.50), "p90": np.quantile(total_resid, 0.90)},
    {"target": "home_margin", "residual_mean": margin_resid.mean(), "residual_std": margin_resid.std(ddof=1), "p10": np.quantile(margin_resid, 0.10), "p50": np.quantile(margin_resid, 0.50), "p90": np.quantile(margin_resid, 0.90)},
])
display(resid_summary)

print()
print("Interpretation:")
print("- residual_mean near 0 means the model is not systematically high/low.")
print("- residual_std is used by the scoring script to approximate over/under and run-line probabilities.")


Totals champion: old_statcast_plus_pitchmix_bullpen__random_forest


,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
15,old_statcast_plus_pitchmix_bullpen__random_forest,standard_regression,4191,1645,3.541386,4.439151,0.009618,8.878443,8.941033,2.221832


Margin champion: old_statcast_plus_pitchmix_bullpen__random_forest


,model_name,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean
1,old_statcast_plus_pitchmix_bullpen__random_forest,4191,1645,3.504739,4.438301,0.036141,0.064521,0.054711


,target,residual_mean,residual_std,p10,p50,p90
0,total_runs,0.06259,4.44006,-5.250380,-0.474545,6.160598
1,home_margin,-0.00981,4.43964,-5.687655,0.265188,5.429851



Interpretation:
- residual_mean near 0 means the model is not systematically high/low.
- residual_std is used by the scoring script to approximate over/under and run-line probabilities.


## 10. Export champion model bundles

Set the approval flags to `True` only after reviewing metrics.

The scoring script uses `residual_std` to turn a predicted total or margin into over/under and run-line cover probabilities.

In [28]:
# Manually select non-leaky champions.
# Do NOT auto-pick totals_results_df.iloc[0] because all_clean totals is leaking final score information.

TOTALS_CHAMPION_NAME = "old_statcast_plus_pitchmix_bullpen__random_forest"
MARGIN_CHAMPION_NAME = "old_statcast_plus_pitchmix_bullpen__random_forest"

assert TOTALS_CHAMPION_NAME in totals_fitted, f"Missing totals model: {TOTALS_CHAMPION_NAME}"
assert MARGIN_CHAMPION_NAME in margin_fitted, f"Missing margin model: {MARGIN_CHAMPION_NAME}"

print("Totals champion:", TOTALS_CHAMPION_NAME)
display(totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)])

print("Margin champion:", MARGIN_CHAMPION_NAME)
display(margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)])

total_resid = y_test_total.values - totals_preds[TOTALS_CHAMPION_NAME]
margin_resid = y_test_margin.values - margin_preds[MARGIN_CHAMPION_NAME]

resid_summary = pd.DataFrame([
    {
        "target": "total_runs",
        "residual_mean": float(np.mean(total_resid)),
        "residual_std": float(np.std(total_resid, ddof=1)),
        "p10": float(np.quantile(total_resid, 0.10)),
        "p50": float(np.quantile(total_resid, 0.50)),
        "p90": float(np.quantile(total_resid, 0.90)),
    },
    {
        "target": "home_margin",
        "residual_mean": float(np.mean(margin_resid)),
        "residual_std": float(np.std(margin_resid, ddof=1)),
        "p10": float(np.quantile(margin_resid, 0.10)),
        "p50": float(np.quantile(margin_resid, 0.50)),
        "p90": float(np.quantile(margin_resid, 0.90)),
    },
])

display(resid_summary)

# Hard guardrail: totals residual std below 2 is a major leakage signal.
assert resid_summary.loc[resid_summary["target"].eq("total_runs"), "residual_std"].iloc[0] > 2.0, \
    "Totals residual std is suspiciously low. Do not export; check leakage."

print()
print("Interpretation:")
print("- residual_mean near 0 means the model is not systematically high/low.")
print("- residual_std is used by the scoring script to approximate over/under and run-line probabilities.")
print("- These residuals look realistic if total_runs residual_std is around 4-ish.")

Totals champion: old_statcast_plus_pitchmix_bullpen__random_forest


,model_name,model_objective,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean,poisson_deviance
15,old_statcast_plus_pitchmix_bullpen__random_forest,standard_regression,4191,1645,3.541386,4.439151,0.009618,8.878443,8.941033,2.221832


Margin champion: old_statcast_plus_pitchmix_bullpen__random_forest


,model_name,feature_count,n_test,mae,rmse,r2,avg_pred,actual_mean
1,old_statcast_plus_pitchmix_bullpen__random_forest,4191,1645,3.504739,4.438301,0.036141,0.064521,0.054711


,target,residual_mean,residual_std,p10,p50,p90
0,total_runs,0.06259,4.44006,-5.250380,-0.474545,6.160598
1,home_margin,-0.00981,4.43964,-5.687655,0.265188,5.429851



Interpretation:
- residual_mean near 0 means the model is not systematically high/low.
- residual_std is used by the scoring script to approximate over/under and run-line probabilities.
- These residuals look realistic if total_runs residual_std is around 4-ish.


In [31]:
APPROVE_TOTALS_EXPORT = True
APPROVE_MARGIN_EXPORT = True

if APPROVE_TOTALS_EXPORT:
    estimator, cols = totals_fitted[TOTALS_CHAMPION_NAME]
    pred = totals_preds[TOTALS_CHAMPION_NAME]
    residual = y_test_total.values - pred
    champion_row = totals_results_df[totals_results_df["model_name"].eq(TOTALS_CHAMPION_NAME)].iloc[0].to_dict()
    bundle = {
        "model_name": TOTALS_CHAMPION_NAME,
        "model_kind": "poisson" if "poisson" in TOTALS_CHAMPION_NAME else "regression",
        "market": "totals",
        "target_col": TARGET_TOTAL_RUNS,
        "model": estimator,
        "feature_cols": cols,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
        "prediction_clip_min": 0.0,
        "residual_mean": float(np.mean(residual)),
        "residual_std": float(np.std(residual, ddof=1)),
        "metrics": champion_row,
    }
    path = MODEL_DIR / "mlb_total_runs_champion.joblib"
    joblib.dump(bundle, path)
    print("Exported", path)
    print("Champion totals bundle model_kind:", bundle["model_kind"])

if APPROVE_MARGIN_EXPORT:
    estimator, cols = margin_fitted[MARGIN_CHAMPION_NAME]
    pred = margin_preds[MARGIN_CHAMPION_NAME]
    residual = y_test_margin.values - pred
    champion_row = margin_results_df[margin_results_df["model_name"].eq(MARGIN_CHAMPION_NAME)].iloc[0].to_dict()
    bundle = {
        "model_name": MARGIN_CHAMPION_NAME,
        "model_kind": "regression",
        "market": "runline",
        "target_col": TARGET_HOME_MARGIN,
        "model": estimator,
        "feature_cols": cols,
        "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "min_train_date": MIN_TRAIN_DATE,
        "residual_mean": float(np.mean(residual)),
        "residual_std": float(np.std(residual, ddof=1)),
        "metrics": champion_row,
    }
    path = MODEL_DIR / "mlb_home_margin_champion.joblib"
    joblib.dump(bundle, path)
    print("Exported", path)


Exported c:\Users\rfo7799\Desktop\Git\TetheredAI\MLB\mlb_betting_app\models\mlb_total_runs_champion.joblib
Champion totals bundle model_kind: regression
Exported c:\Users\rfo7799\Desktop\Git\TetheredAI\MLB\mlb_betting_app\models\mlb_home_margin_champion.joblib


## 11. Odds prerequisite check for production

Before totals/run-line scoring can generate recommendations, the Odds API table must contain `totals` and `spreads` markets, not just `h2h`.

Run this locally/Cloud Shell after a daily odds fetch:

```sql
SELECT market_key, COUNT(*) FROM odds_snapshots GROUP BY market_key;
```

If only `h2h` appears, update the odds fetch job/script to request `h2h,spreads,totals`.

In [32]:
# Optional local DB market inspection.
import sqlite3
DB_PATH = PROJECT_ROOT / "data" / "odds.db"
if DB_PATH.exists() and DB_PATH.stat().st_size > 0:
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        markets = pd.read_sql_query("SELECT market_key, COUNT(*) AS rows FROM odds_snapshots GROUP BY market_key ORDER BY rows DESC", conn)
        display(markets)
    finally:
        conn.close()
else:
    print("No local odds.db found; skip market inspection.")

No local odds.db found; skip market inspection.
